# OMIn GraphRAG: 5-Graph Construction

**Goal:** Build a clean NetworkX knowledge graph from the extracted triplets.

**Steps:**
1. Load triplets + embeddings from notebook 2
2. Entity deduplication via embedding similarity clustering : used threshold 0.92 which is fine, a smaller one may also do a good job, for example 0.88
3. Build graph: flat weight=1, full record_id list per edge
4. Compute and store graph schema (used by guardrail in notebook 6)
5. Serialize graph to GML + stats

**Design decisions:**
- Edge weight = 1 (flat) ==> with semantic similarity between query and graph entities/relation will learn relevance, not frequency
- Each edge stores full list of record_ids that support it
- Each node stores its embedding vector for query-time retrieval
- Deduplication: cosine similarity > 0.92 threshold merges entity variants

## Setup

In [ ]:
!pip install -q networkx numpy scikit-learn tqdm matplotlib

In [ ]:
import json, os
import numpy as np
import networkx as nx
from collections import defaultdict
from tqdm import tqdm
import matplotlib.pyplot as plt

os.makedirs("./output", exist_ok=True)

# Load all outputs from notebook 2
with open("./output/triplets.json") as f:
    triplets = json.load(f)

with open("./output/entity_embeddings.json") as f:
    entity_embeddings = json.load(f)

with open("./output/relation_embeddings.json") as f:
    relation_embeddings = json.load(f)

with open("./output/records.json") as f:
    records = json.load(f)

# Build record lookup: record_id -> record
record_lookup = {r["record_id"]: r for r in records}

print(f"Triplets loaded:          {len(triplets)}")
print(f"Entity embeddings loaded: {len(entity_embeddings)}")
print(f"Records loaded:           {len(records)}")

RELATION_TYPES = [
    "CAUSES", "RESULTS_IN", "PART_OF", "LOCATED_IN",
    "REQUIRES_ACTION", "INSPECTED_BY", "REPLACED_BY",
    "PRECEDED_BY", "AFFECTS", "INSTANCE_OF"
]

## Entity deduplication via embedding similarity

Merges near-duplicate entity variants:
- 'fuel pump' / 'fuel pumps' / 'aircraft fuel pump' → canonical form
- 'engine failure' / 'engine fail' → canonical form

Strategy: cosine similarity > 0.92 → same entity, keep the shortest clean name as canonical.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

DEDUP_THRESHOLD = 0.92  # tune if too aggressive or too loose

# Build embedding matrix
entities = list(entity_embeddings.keys())
emb_matrix = np.array([entity_embeddings[e] for e in entities], dtype=np.float32)

print(f"Building similarity matrix for {len(entities)} entities...")

# Process in chunks to avoid memory issues
CHUNK = 500
entity_to_cluster = {}   # entity -> cluster_id
cluster_members   = defaultdict(list)  # cluster_id -> [entities]
next_cluster_id   = 0

for i in tqdm(range(0, len(entities), CHUNK), desc="Clustering"):
    chunk_entities = entities[i:i+CHUNK]
    chunk_embs     = emb_matrix[i:i+CHUNK]

    # Compare this chunk against all already-processed entities
    for j, (entity, emb) in enumerate(zip(chunk_entities, chunk_embs)):
        if entity in entity_to_cluster:
            continue

        # Compare against existing cluster representatives
        assigned = False
        for cid, members in cluster_members.items():
            rep_emb = np.array(entity_embeddings[members[0]], dtype=np.float32)
            sim = float(np.dot(emb, rep_emb) / (np.linalg.norm(emb) * np.linalg.norm(rep_emb) + 1e-9))
            if sim >= DEDUP_THRESHOLD:
                entity_to_cluster[entity] = cid
                cluster_members[cid].append(entity)
                assigned = True
                break

        if not assigned:
            entity_to_cluster[entity] = next_cluster_id
            cluster_members[next_cluster_id].append(entity)
            next_cluster_id += 1

print(f"\nOriginal entities:    {len(entities)}")
print(f"Clusters (canonical): {len(cluster_members)}")
print(f"Reduction:            {len(entities) - len(cluster_members)} duplicates merged")

# Show sample merges
print("\nSAMPLE MERGED CLUSTERS")
shown = 0
for cid, members in cluster_members.items():
    if len(members) > 1:
        print(f"  Cluster {cid}: {members}")
        shown += 1
        if shown >= 15:
            break

In [ ]:
# Build canonical name map: entity -> canonical_name
# Canonical = shortest member (usually cleanest)
canonical_map = {}
canonical_embeddings = {}  # canonical_name -> embedding (average of cluster)

for cid, members in cluster_members.items():
    # Pick shortest name as canonical (ties broken by alphabetical)
    canonical = sorted(members, key=lambda x: (len(x), x))[0]
    for m in members:
        canonical_map[m] = canonical
    # Average embedding across cluster members
    cluster_embs = np.array([entity_embeddings[m] for m in members], dtype=np.float32)
    canonical_embeddings[canonical] = cluster_embs.mean(axis=0).tolist()

# Save canonical embeddings (replaces entity_embeddings for graph use)
with open("./output/canonical_embeddings.json", "w") as f:
    json.dump(canonical_embeddings, f)

with open("./output/canonical_map.json", "w") as f:
    json.dump(canonical_map, f)

print(f"Canonical entity map saved: {len(canonical_map)} entities -> {len(canonical_embeddings)} canonical nodes")

# Apply canonical map to triplets
normalized_triplets = []
for t in triplets:
    head = canonical_map.get(t["head"], t["head"])
    tail = canonical_map.get(t["tail"], t["tail"])
    if head != tail:  # skip self-loops
        normalized_triplets.append({
            "head":      head,
            "relation":  t["relation"],
            "tail":      tail,
            "record_id": t["record_id"]
        })

print(f"Triplets after normalization: {len(normalized_triplets)} (was {len(triplets)})")

## Build the Knowledge Graph

**Node attributes:**
- `label`: canonical entity name
- `embedding`: Gemini embedding vector (for query-time similarity)
- `record_ids`: list of records that mention this entity
- `frequency`: how many records mention this entity

**Edge attributes:**
- `relation`: one of our 10 relation types
- `weight`: 1.0 (flat, with semantic similarity, it learns relevance)
- `record_ids`: list of records that support this specific triplet
- `support_count`: len(record_ids) ==> useful for analysis

Graph type: **MultiDiGraph** (directed, allows multiple edges between same nodes with different relations)

In [ ]:
G = nx.MultiDiGraph()

# Build edge index first: (head, relation, tail) -> [record_ids]
edge_records = defaultdict(list)   # (head, relation, tail) -> [record_id]
node_records = defaultdict(set)    # entity -> {record_ids}

for t in normalized_triplets:
    key = (t["head"], t["relation"], t["tail"])
    edge_records[key].append(t["record_id"])
    node_records[t["head"]].add(t["record_id"])
    node_records[t["tail"]].add(t["record_id"])

print(f"Unique (head, relation, tail) triples: {len(edge_records)}")
print(f"Unique nodes (canonical entities):     {len(node_records)}")

# Add nodes
for entity, rec_ids in tqdm(node_records.items(), desc="Adding nodes"):
    emb = canonical_embeddings.get(entity, [])
    G.add_node(
        entity,
        label=entity,
        embedding=emb,
        record_ids=list(rec_ids),
        frequency=len(rec_ids)
    )

# Add edges
for (head, relation, tail), rec_ids in tqdm(edge_records.items(), desc="Adding edges"):
    # Deduplicate record_ids
    unique_recs = list(set(rec_ids))
    G.add_edge(
        head, tail,
        relation=relation,
        weight=1.0,
        record_ids=unique_recs,
        support_count=len(unique_recs)
    )

print(f"\n*** GRAPH BUILT ***")
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print(f"Avg edges per node: {G.number_of_edges() / max(G.number_of_nodes(), 1):.2f}")

## Graph statistics and sanity checks

In [ ]:
from collections import Counter

# Relation distribution in graph
relation_counts = Counter()
support_counts  = []
for u, v, data in G.edges(data=True):
    relation_counts[data['relation']] += 1
    support_counts.append(data['support_count'])

print("=== RELATION DISTRIBUTION IN GRAPH ===")
for rel, count in sorted(relation_counts.items(), key=lambda x: -x[1]):
    print(f"  {rel:<20} {count:>6} edges")

print(f"\n=== EDGE SUPPORT STATS ===")
print(f"  Edges with 1 supporting record:   {sum(1 for c in support_counts if c==1)}")
print(f"  Edges with 2-5 supporting records: {sum(1 for c in support_counts if 2<=c<=5)}")
print(f"  Edges with >5 supporting records:  {sum(1 for c in support_counts if c>5)}")
print(f"  Max support count: {max(support_counts)}")
print(f"  Avg support count: {np.mean(support_counts):.2f}")

# Top nodes by degree
degree_seq = sorted(dict(G.degree()).items(), key=lambda x: -x[1])
print(f"\n=== TOP 20 NODES BY DEGREE ===")
for node, deg in degree_seq[:20]:
    print(f"  {node:<40} degree={deg}")

# Connectivity
undirected = G.to_undirected()
components = list(nx.connected_components(undirected))
print(f"\n=== CONNECTIVITY ===")
print(f"  Connected components: {len(components)}")
print(f"  Largest component:    {max(len(c) for c in components)} nodes")
print(f"  Isolated nodes:       {sum(1 for c in components if len(c)==1)}")

## Build and save the Graph Schema

The schema is used by the **pre-generation guardrail**.
It captures:
- Which entity types exist in the graph
- Which relation types exist between which entity clusters
- How densely connected each relation type is
- What the graph 'knows about' (high-degree hub entities)

This schema is what the guardrail LLM reads to generate the 'ideal subschema' for a query.

In [ ]:
# Build graph schema descriptor
schema = {
    "graph_stats": {
        "total_nodes": G.number_of_nodes(),
        "total_edges": G.number_of_edges(),
        "total_records": len(records),
        "relation_types": RELATION_TYPES
    },
    "relation_distribution": dict(relation_counts),
    "hub_entities": [
        {"entity": node, "degree": deg, "record_count": G.nodes[node].get("frequency", 0)}
        for node, deg in degree_seq[:50]  # top 50 hub nodes
    ],
    "relation_patterns": {}  # which relations connect which types of entities
}

# For each relation type, collect sample (head, tail) pairs
for rel in RELATION_TYPES:
    samples = []
    for u, v, data in G.edges(data=True):
        if data['relation'] == rel:
            samples.append({
                "head": u,
                "tail": v,
                "support": data['support_count']
            })
    # Sort by support count, keep top 20 as schema examples
    samples = sorted(samples, key=lambda x: -x['support'])[:20]
    schema["relation_patterns"][rel] = {
        "count": relation_counts[rel],
        "top_examples": samples
    }

with open("./output/graph_schema.json", "w") as f:
    json.dump(schema, f, indent=2)

print("Graph schema saved to ./output/graph_schema.json")
print("\nSchema summary:")
print(f"  Nodes: {schema['graph_stats']['total_nodes']}")
print(f"  Edges: {schema['graph_stats']['total_edges']}")
print(f"  Hub entities (top 5): {[h['entity'] for h in schema['hub_entities'][:5]]}")

## Debug :  serialize the graph

In [ ]:
# GML cannot store lists natively — serialize list attributes to JSON strings
G_export = G.copy()

for node, data in G_export.nodes(data=True):
    data['record_ids'] = json.dumps(data.get('record_ids', []))
    data['embedding']  = json.dumps(data.get('embedding', []))

for u, v, key, data in G_export.edges(data=True, keys=True):
    data['record_ids'] = json.dumps(data.get('record_ids', []))

nx.write_gml(G_export, "./output/knowledge_graph.gml")
print("Graph saved to ./output/knowledge_graph.gml")

# Also save as JSON for easier inspection
graph_json = {
    "nodes": [
        {
            "id": node,
            "record_ids": data.get("record_ids", []),
            "frequency": data.get("frequency", 0)
        }
        for node, data in G.nodes(data=True)
    ],
    "edges": [
        {
            "head": u,
            "relation": data["relation"],
            "tail": v,
            "weight": data["weight"],
            "record_ids": data["record_ids"],
            "support_count": data["support_count"]
        }
        for u, v, data in G.edges(data=True)
    ]
}

with open("./output/knowledge_graph.json", "w") as f:
    json.dump(graph_json, f, indent=2)

print("Graph saved to ./output/knowledge_graph.json")

# Quick size check
gml_size  = os.path.getsize("./output/knowledge_graph.gml")  / 1024
json_size = os.path.getsize("./output/knowledge_graph.json") / 1024
print(f"GML size:  {gml_size:.1f} KB")
print(f"JSON size: {json_size:.1f} KB")

## 7. Visualize a small subgraph (sanity check)

In [ ]:
# Pick the top hub node and show its 1-hop neighborhood
hub = degree_seq[0][0]
print(f"Visualizing 1-hop neighborhood of hub node: '{hub}'")

neighbors  = list(G.predecessors(hub)) + list(G.successors(hub))
subgraph_nodes = [hub] + neighbors[:20]  # cap at 20 for readability
sub = G.subgraph(subgraph_nodes)

plt.figure(figsize=(14, 9))
pos = nx.spring_layout(sub, seed=42, k=2)

# Color hub node differently
node_colors = ["#e74c3c" if n == hub else "#3498db" for n in sub.nodes()]
nx.draw_networkx_nodes(sub, pos, node_color=node_colors, node_size=800, alpha=0.9)
nx.draw_networkx_labels(sub, pos, font_size=7, font_color="white", font_weight="bold")

# Draw edges with relation labels
edge_labels = {(u, v): data['relation'] for u, v, data in sub.edges(data=True)}
nx.draw_networkx_edges(sub, pos, arrows=True, arrowsize=15,
                       edge_color="#7f8c8d", width=1.5, alpha=0.7,
                       connectionstyle="arc3,rad=0.1")
nx.draw_networkx_edge_labels(sub, pos, edge_labels=edge_labels,
                              font_size=6, font_color="#2c3e50")

plt.title(f"1-hop subgraph around '{hub}' (red=hub, blue=neighbors)", fontsize=12)
plt.axis("off")
plt.tight_layout()
plt.savefig("./output/subgraph_viz.png", dpi=150, bbox_inches="tight")
plt.show()
print("Visualization saved to ./output/subgraph_viz.png")